# 03 — Calibration Analysis

**Purpose**: Evaluate how well-calibrated Polymarket equity prediction markets are as probabilistic forecasts.

**Key metrics**:
- Reliability diagrams
- Brier score and decomposition (reliability, resolution, uncertainty)
- Expected Calibration Error (ECE)
- Log-loss

**Subgroup analyses**: by market type, volume/liquidity, ticker

**Outputs**: Calibration tables + plots in `results/`

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

import config as cfg
import thesis_utils as tu

data_dir, results_dir = tu.ensure_project_dirs(cfg.PROJECT_ROOT)
figures_dir = cfg.FIGURES_DIR
figures_dir.mkdir(parents=True, exist_ok=True)

## 1. Load Resolved Markets

In [ ]:
df = tu.load_parquet('market_resolutions.parquet')
print(f'Resolved markets: {len(df):,}')
print(f'Columns: {list(df.columns)}')

# For calibration we need: implied_prob (predicted) and outcome_int (actual)
# Filter to markets with both
df_cal = df[df['implied_prob'].notna() & df['outcome_int'].notna()].copy()
print(f'\nMarkets with both implied_prob and outcome: {len(df_cal):,}')

# If no implied_prob available, we can still do outcome-only analysis
# and report calibration once prices are collected
if len(df_cal) < 100:
    print('\nWARNING: Too few markets with implied probability for meaningful calibration.')
    print('Run 01_data_collection.ipynb first to fetch CLOB/Gamma prices.')
    print(f'\nFalling back to outcome-only analysis with {len(df):,} resolved markets.')

## 2. Overall Calibration

In [ ]:
if len(df_cal) >= 100:
    predicted = df_cal['implied_prob'].values
    observed = df_cal['outcome_int'].values.astype(float)

    # Brier score
    bs = tu.brier_score(predicted, observed)
    print(f'Brier Score: {bs:.4f}')

    # Brier decomposition
    decomp = tu.brier_decomposition(predicted, observed, n_bins=cfg.CALIBRATION_BINS)
    print(f'  Reliability:  {decomp["reliability"]:.4f}  (lower = better calibrated)')
    print(f'  Resolution:   {decomp["resolution"]:.4f}  (higher = more discriminating)')
    print(f'  Uncertainty:  {decomp["uncertainty"]:.4f}  (base rate entropy)')

    # ECE
    ece = tu.expected_calibration_error(predicted, observed, n_bins=cfg.CALIBRATION_BINS)
    print(f'\nExpected Calibration Error (ECE): {ece:.4f}')

    # Log-loss
    ll = tu.log_loss(predicted, observed)
    print(f'Log-Loss: {ll:.4f}')

    # Bin details table
    bin_df = pd.DataFrame(decomp['bin_details'])
    print(f'\n=== Calibration Bins ===')
    bin_df
else:
    print('Insufficient data for full calibration — see fallback analysis below.')

In [ ]:
def plot_reliability_diagram(predicted, observed, title='Reliability Diagram',
                             n_bins=10, ax=None, label=None, color='steelblue'):
    """Plot a reliability (calibration) diagram."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))

    # Compute calibration curve
    prob_true, prob_pred = calibration_curve(observed, predicted, n_bins=n_bins, strategy='uniform')

    # Perfect calibration line
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')

    # Calibration curve
    ax.plot(prob_pred, prob_true, 's-', color=color, label=label or 'Polymarket', markersize=8)

    # Histogram of predictions (secondary y-axis)
    ax2 = ax.twinx()
    ax2.hist(predicted, bins=n_bins, range=(0, 1), alpha=0.15, color=color)
    ax2.set_ylabel('Count', color='gray', alpha=0.7)
    ax2.tick_params(axis='y', labelcolor='gray', alpha=0.5)

    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives (Observed)')
    ax.set_title(title)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

    return ax

if len(df_cal) >= 100:
    fig, ax = plt.subplots(figsize=(7, 7))
    plot_reliability_diagram(predicted, observed, 
                            title=f'Overall Calibration (N={len(df_cal):,}, Brier={bs:.3f}, ECE={ece:.3f})',
                            ax=ax)
    plt.tight_layout()
    plt.savefig(figures_dir / 'calibration_overall.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
    plt.show()

## 3. Calibration by Market Type

In [ ]:
if len(df_cal) >= 100:
    type_metrics = []
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    market_types = df_cal['market_type'].value_counts()
    plot_types = [mt for mt, ct in market_types.items() if ct >= cfg.MIN_BIN_COUNT]

    for i, mtype in enumerate(plot_types[:6]):
        subset = df_cal[df_cal['market_type'] == mtype]
        pred_sub = subset['implied_prob'].values
        obs_sub = subset['outcome_int'].values.astype(float)

        bs_sub = tu.brier_score(pred_sub, obs_sub)
        ece_sub = tu.expected_calibration_error(pred_sub, obs_sub)

        type_metrics.append({
            'market_type': mtype,
            'n': len(subset),
            'brier': bs_sub,
            'ece': ece_sub,
            'base_rate': obs_sub.mean(),
        })

        plot_reliability_diagram(
            pred_sub, obs_sub,
            title=f'{mtype} (N={len(subset)}, BS={bs_sub:.3f})',
            ax=axes[i],
        )

    # Hide unused axes
    for j in range(len(plot_types), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Calibration by Market Type', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(figures_dir / 'calibration_by_type.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
    plt.show()

    type_metrics_df = pd.DataFrame(type_metrics).sort_values('n', ascending=False)
    type_metrics_df

## 4. Calibration by Liquidity (Volume)

In [ ]:
if len(df_cal) >= 100:
    # Split by volume terciles
    df_cal['volume_group'] = pd.qcut(
        df_cal['volume'].fillna(0), q=3,
        labels=['Low Volume', 'Medium Volume', 'High Volume']
    )

    vol_metrics = []
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    colors = ['#e74c3c', '#f39c12', '#27ae60']

    for i, vgroup in enumerate(['Low Volume', 'Medium Volume', 'High Volume']):
        subset = df_cal[df_cal['volume_group'] == vgroup]
        if len(subset) < cfg.MIN_BIN_COUNT:
            continue
        pred_sub = subset['implied_prob'].values
        obs_sub = subset['outcome_int'].values.astype(float)

        bs_sub = tu.brier_score(pred_sub, obs_sub)
        ece_sub = tu.expected_calibration_error(pred_sub, obs_sub)
        vol_range = f'${subset["volume"].min():,.0f}–${subset["volume"].max():,.0f}'

        vol_metrics.append({
            'volume_group': vgroup,
            'n': len(subset),
            'brier': bs_sub,
            'ece': ece_sub,
            'volume_range': vol_range,
        })

        plot_reliability_diagram(
            pred_sub, obs_sub,
            title=f'{vgroup} (N={len(subset)}, BS={bs_sub:.3f})',
            ax=axes[i], color=colors[i],
        )

    plt.suptitle('Calibration by Volume Tercile', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(figures_dir / 'calibration_by_volume.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
    plt.show()

    pd.DataFrame(vol_metrics)

## 5. Calibration by Ticker

In [ ]:
if len(df_cal) >= 100:
    ticker_metrics = []
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i, ticker in enumerate(cfg.PRIMARY_TICKERS):
        subset = df_cal[df_cal['ticker'] == ticker]
        if len(subset) < cfg.MIN_BIN_COUNT:
            print(f'  {ticker}: only {len(subset)} markets — skipping')
            continue
        pred_sub = subset['implied_prob'].values
        obs_sub = subset['outcome_int'].values.astype(float)

        bs_sub = tu.brier_score(pred_sub, obs_sub)
        ece_sub = tu.expected_calibration_error(pred_sub, obs_sub)

        ticker_metrics.append({
            'ticker': ticker,
            'n': len(subset),
            'brier': bs_sub,
            'ece': ece_sub,
            'base_rate': obs_sub.mean(),
        })

        color = cfg.TICKER_COLORS.get(ticker, 'steelblue')
        plot_reliability_diagram(
            pred_sub, obs_sub,
            title=f'{ticker} (N={len(subset)}, BS={bs_sub:.3f})',
            ax=axes[i], color=color,
        )

    plt.suptitle('Calibration by Ticker', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(figures_dir / 'calibration_by_ticker.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
    plt.show()

    pd.DataFrame(ticker_metrics)

## 6. Summary Table

In [ ]:
# Combine all metrics into a summary
if len(df_cal) >= 100:
    overall_row = pd.DataFrame([{
        'group': 'Overall',
        'n': len(df_cal),
        'brier': bs,
        'ece': ece,
        'log_loss': ll,
        'base_rate': observed.mean(),
    }])

    summary = overall_row.copy()

    if type_metrics:
        tm = pd.DataFrame(type_metrics)
        tm['group'] = 'Type: ' + tm['market_type']
        summary = pd.concat([summary, tm[['group', 'n', 'brier', 'ece', 'base_rate']]], ignore_index=True)

    if ticker_metrics:
        tkm = pd.DataFrame(ticker_metrics)
        tkm['group'] = 'Ticker: ' + tkm['ticker']
        summary = pd.concat([summary, tkm[['group', 'n', 'brier', 'ece', 'base_rate']]], ignore_index=True)

    # Save
    summary.to_csv(results_dir / 'calibration_summary.csv', index=False)
    print('Saved calibration_summary.csv')
    summary
else:
    # Outcome-only summary (no implied probabilities)
    print('=== Outcome-Only Summary (no implied probabilities) ===')
    df_valid = df[df['outcome_int'].notna()]
    summary_outcome = df_valid.groupby('market_type').agg(
        n=('outcome_int', 'size'),
        yes_rate=('outcome_int', 'mean'),
        avg_volume=('volume', 'mean'),
    ).sort_values('n', ascending=False)
    summary_outcome.to_csv(results_dir / 'outcome_summary.csv')
    summary_outcome